# Xarray Chunking Best Practices - dask_setup Recipe

This notebook demonstrates intelligent chunking strategies for xarray datasets, implementing the 256-512 MiB per chunk rule and comparing different approaches for optimal memory usage and performance.

## Key Learning Points
- How to choose optimal chunk sizes for your data
- When to use `.chunk()` vs `rechunk_dataset()`
- Memory-efficient chunking patterns
- Impact of chunk size on performance
- Bad vs good chunking patterns

## Setup and Imports

Import necessary libraries and configure the environment.

In [ ]:
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Add dask_setup to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
import numpy as np
import dask
import dask.array as da
import xarray as xr

# Import dask_setup
try:
    from dask_setup import setup_dask_client
    print("✅ dask_setup imported successfully")
except ImportError as e:
    print(f"❌ Failed to import dask_setup: {e}")
    
# Try to import utilities
try:
    from utils import format_bytes, format_duration, timer
    print("✅ utils imported successfully")
except ImportError as e:
    print(f"⚠️  utils not available: {e}")
    # Define minimal versions
    def format_bytes(bytes_val):
        return f"{bytes_val / (1024**3):.2f} GB"
    def format_duration(seconds):
        return f"{seconds:.2f}s"
    from contextlib import contextmanager
    import time
    @contextmanager
    def timer(name):
        start = time.time()
        result = {'total': 0}
        yield result
        result['total'] = time.time() - start

# Optional dependencies
try:
    import rechunker
    RECHUNKER_AVAILABLE = True
    print("✅ rechunker available")
except ImportError:
    RECHUNKER_AVAILABLE = False
    print("⚠️  rechunker not available - some features will be skipped")

try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
    print("✅ matplotlib available for plotting")
except ImportError:
    PLOTTING_AVAILABLE = False
    print("⚠️  matplotlib not available - plotting disabled")

print("\n🚀 Setup complete!")

## Create Sample Dataset

Let's create a sample climate-like dataset to work with.

In [ ]:
def create_sample_dataset(shape=(365, 180, 360), dims=('time', 'lat', 'lon'), variables=None):
    """Create a sample xarray dataset for chunking demonstrations."""
    if variables is None:
        variables = ['temperature', 'precipitation']
    
    print(f"Creating sample dataset with shape: {shape}")
    print(f"Dimensions: {dims}")
    print(f"Variables: {variables}")
    
    # Create coordinates
    coords = {}
    if 'time' in dims:
        time_idx = dims.index('time')
        coords['time'] = np.arange('2000-01-01', 
                                 f'2000-01-{shape[time_idx] + 1}', dtype='datetime64[D]')
    
    if 'lat' in dims:
        lat_idx = dims.index('lat')
        coords['lat'] = np.linspace(-90, 90, shape[lat_idx])
        
    if 'lon' in dims:
        lon_idx = dims.index('lon')
        coords['lon'] = np.linspace(-180, 180, shape[lon_idx])
    
    # Create data variables
    data_vars = {}
    for var in variables:
        # Create realistic-looking data
        if var == 'temperature':
            data = da.random.random(shape, chunks=None) * 50 - 10 + 273.15  # Kelvin
            attrs = {'units': 'K', 'long_name': 'Near-surface air temperature'}
        elif var == 'precipitation':
            data = da.random.exponential(scale=2.0, size=shape, chunks=None)  # mm/day
            attrs = {'units': 'mm/day', 'long_name': 'Daily precipitation'}
        else:
            data = da.random.random(shape, chunks=None)
            attrs = {'units': 'unknown', 'long_name': f'Sample variable {var}'}
        
        data_vars[var] = (dims, data, attrs)
    
    # Create dataset
    ds = xr.Dataset(data_vars, coords=coords)
    
    # Add global attributes
    ds.attrs['title'] = 'Sample climate dataset for chunking demonstrations'
    ds.attrs['created_by'] = 'dask_setup xarray chunking recipe'
    
    return ds

# Create our sample dataset
dataset_shape = (365, 180, 360)  # One year of daily global data
ds = create_sample_dataset(
    shape=dataset_shape,
    dims=('time', 'lat', 'lon'),
    variables=['temperature', 'precipitation']
)

print(f"\n📊 Dataset Information:")
print(f"   Shape: {ds.temperature.shape}")
print(f"   Size: {ds.nbytes / (1024**2):.1f} MB")
print(f"   Variables: {list(ds.data_vars.keys())}")
print(f"   Coordinates: {list(ds.coords.keys())}")

# Display the dataset
print("\nDataset summary:")
print(ds)

## Optimal Chunk Size Calculator

Let's create a function to calculate optimal chunk sizes based on the 256-512 MB rule and workload type.

In [ ]:
def calculate_optimal_chunks(
    ds: xr.Dataset,
    target_chunk_mb: Tuple[float, float] = (256, 512),
    workload_type: str = "balanced"
) -> Dict[str, int]:
    """Calculate optimal chunk sizes based on the 256-512 MiB rule.
    
    Args:
        ds: xarray Dataset to analyze
        target_chunk_mb: Target chunk size range in MB
        workload_type: Type of workload ("cpu", "io", "balanced")
        
    Returns:
        Dictionary with recommended chunk sizes per dimension
    """
    print(f"\n🎯 Calculating optimal chunks for {workload_type} workload")
    print(f"   Target chunk size: {target_chunk_mb[0]}-{target_chunk_mb[1]} MB")
    
    # Get the largest data variable for analysis
    largest_var = max(ds.data_vars.values(), key=lambda v: v.nbytes)
    
    print(f"   Analyzing variable: {largest_var.name}")
    print(f"   Shape: {largest_var.shape}")
    print(f"   Dimensions: {largest_var.dims}")
    
    # Calculate bytes per element
    dtype_size = np.dtype(largest_var.dtype).itemsize
    print(f"   Data type: {largest_var.dtype} ({dtype_size} bytes/element)")
    
    # Target chunk size in bytes
    target_min_bytes = target_chunk_mb[0] * 1024 * 1024
    target_max_bytes = target_chunk_mb[1] * 1024 * 1024
    target_bytes = (target_min_bytes + target_max_bytes) / 2
    
    # Calculate target number of elements per chunk
    target_elements = target_bytes / dtype_size
    
    # Strategy based on workload type
    chunks = {}
    
    if workload_type == "cpu":
        print("   Strategy: Square-ish chunks for CPU optimization")
        total_dims = len(largest_var.dims)
        elements_per_dim = int(target_elements ** (1.0 / total_dims))
        
        for dim in largest_var.dims:
            dim_size = largest_var.sizes[dim]
            chunk_size = min(dim_size, max(1, elements_per_dim))
            chunks[dim] = chunk_size
            
    elif workload_type == "io":
        print("   Strategy: I/O-friendly chunks (preserve record dimensions)")
        
        # Identify likely record dimensions (time, often the first)
        record_dims = []
        for dim in largest_var.dims:
            if 'time' in dim.lower() or largest_var.dims.index(dim) == 0:
                record_dims.append(dim)
        
        remaining_elements = target_elements
        non_record_dims = [d for d in largest_var.dims if d not in record_dims]
        
        for dim in largest_var.dims:
            dim_size = largest_var.sizes[dim]
            if dim in record_dims:
                chunks[dim] = min(dim_size, max(100, dim_size // 4))
            else:
                if non_record_dims:
                    elements_per_spatial = int(remaining_elements ** (1.0 / len(non_record_dims)))
                    chunk_size = min(dim_size, max(1, elements_per_spatial))
                    chunks[dim] = chunk_size
                else:
                    chunks[dim] = dim_size
                    
    else:  # balanced
        print("   Strategy: Balanced chunks")
        total_elements = np.prod(largest_var.shape)
        
        for dim in largest_var.dims:
            dim_size = largest_var.sizes[dim]
            dim_fraction = dim_size / total_elements ** (1.0 / len(largest_var.dims))
            chunk_size = int(target_elements ** (1.0 / len(largest_var.dims)) * dim_fraction)
            chunk_size = min(dim_size, max(1, chunk_size))
            chunks[dim] = chunk_size
    
    # Refine chunks to hit target size more precisely
    current_elements = np.prod(list(chunks.values()))
    current_bytes = current_elements * dtype_size
    
    if current_bytes < target_min_bytes or current_bytes > target_max_bytes:
        adjustment_factor = (target_bytes / current_bytes) ** (1.0 / len(chunks))
        
        for dim in chunks:
            dim_size = largest_var.sizes[dim]
            chunks[dim] = min(dim_size, max(1, int(chunks[dim] * adjustment_factor)))
    
    # Final validation
    final_elements = np.prod(list(chunks.values()))
    final_bytes = final_elements * dtype_size
    final_mb = final_bytes / (1024 * 1024)
    
    print(f"   Recommended chunks: {chunks}")
    print(f"   Estimated chunk size: {final_mb:.1f} MB")
    
    return chunks


# Calculate optimal chunks for different workload types
chunk_strategies = {}

for workload in ['cpu', 'io', 'balanced']:
    chunks = calculate_optimal_chunks(ds, workload_type=workload)
    chunk_strategies[f"optimal_{workload}"] = chunks

print("\n📋 Summary of optimal chunking strategies:")
for strategy, chunks in chunk_strategies.items():
    print(f"   {strategy}: {chunks}")

## Chunking Analysis and Comparison

Let's analyze different chunking options and compare their properties.

In [ ]:
def estimate_chunk_memory(ds, chunks=None):
    """Estimate memory usage for each variable in the dataset."""
    estimates = {}
    
    for var_name, var in ds.data_vars.items():
        if chunks:
            # Apply chunking to get chunk info
            var_chunked = var.chunk(chunks)
            if hasattr(var_chunked.data, 'chunks'):
                # Calculate actual chunk sizes
                chunk_shapes = [tuple(c[0] if len(c) == 1 else max(c) for c in var_chunked.data.chunks)]
                if chunk_shapes:
                    chunk_elements = np.prod(chunk_shapes[0])
                    chunk_bytes = chunk_elements * np.dtype(var.dtype).itemsize
                    chunk_mb = chunk_bytes / (1024**2)
                    total_chunks = var_chunked.data.nchunks
                    total_mb = chunk_mb * total_chunks
                else:
                    chunk_mb = var.nbytes / (1024**2)
                    total_chunks = 1
                    total_mb = chunk_mb
            else:
                chunk_mb = var.nbytes / (1024**2)
                total_chunks = 1 
                total_mb = chunk_mb
        else:
            # No chunking - single chunk
            chunk_mb = var.nbytes / (1024**2)
            total_chunks = 1
            total_mb = chunk_mb
        
        estimates[var_name] = {
            'chunk_mb': chunk_mb,
            'total_chunks': total_chunks,
            'total_mb': total_mb
        }
    
    return estimates


def analyze_chunking_options(ds: xr.Dataset, custom_chunks=None) -> Dict[str, Dict]:
    """Analyze different chunking options for a dataset."""
    
    # Define chunking options to test
    chunk_options = {
        **chunk_strategies,  # Our calculated optimal chunks
        "small_chunks": {dim: min(100, size // 10) for dim, size in ds.sizes.items()},
        "large_chunks": {dim: min(1000, size // 2) for dim, size in ds.sizes.items()},
        "time_continuous": {'time': ds.sizes['time'], 'lat': 90, 'lon': 180},
        "spatial_focus": {'time': 30, 'lat': 180, 'lon': 360}
    }
    
    if custom_chunks:
        chunk_options['custom'] = custom_chunks
    
    print("\n" + "=" * 60)
    print("📊 CHUNKING OPTIONS ANALYSIS")
    print("=" * 60)
    
    results = {}
    
    for option_name, chunks in chunk_options.items():
        print(f"\n🔍 Analyzing: {option_name}")
        
        try:
            # Estimate memory usage
            memory_estimates = estimate_chunk_memory(ds, chunks)
            
            # Calculate totals
            total_chunks = sum(est['total_chunks'] for est in memory_estimates.values())
            total_memory_mb = sum(est['total_mb'] for est in memory_estimates.values())
            avg_chunk_mb = np.mean([est['chunk_mb'] for est in memory_estimates.values()])
            
            # Assess chunk size quality
            if 256 <= avg_chunk_mb <= 512:
                quality = "✅ Optimal"
            elif 128 <= avg_chunk_mb <= 1024:
                quality = "🟡 Acceptable"
            elif avg_chunk_mb < 128:
                quality = "🔴 Too small (overhead issues)"
            else:
                quality = "🔴 Too large (memory issues)"
            
            results[option_name] = {
                'chunks': chunks,
                'total_chunks': total_chunks,
                'total_memory_mb': total_memory_mb,
                'avg_chunk_mb': avg_chunk_mb,
                'quality': quality,
                'memory_estimates': memory_estimates
            }
            
            print(f"   Chunks: {chunks}")
            print(f"   Total chunks: {total_chunks}")
            print(f"   Avg chunk size: {avg_chunk_mb:.1f} MB")
            print(f"   Total memory: {total_memory_mb:.1f} MB")
            print(f"   Quality: {quality}")
            
        except Exception as e:
            print(f"   ❌ Analysis failed: {e}")
            results[option_name] = {
                'error': str(e),
                'quality': '❌ Failed'
            }
    
    return results

# Analyze our chunking options
analysis_results = analyze_chunking_options(ds)

print("\n📋 Analysis Summary:")
for strategy, result in analysis_results.items():
    if 'error' not in result:
        print(f"   {strategy:20s} | {result['avg_chunk_mb']:6.1f} MB | {result['quality']}")

## Performance Benchmarking

Let's benchmark the performance of different chunking strategies using actual computations.

In [ ]:
# Set up a Dask client for benchmarking
client, cluster, temp_dir = setup_dask_client(
    workload_type="mixed",
    max_workers=4,
    reserve_mem_gb=20.0,
    dashboard=True
)

print(f"✅ Cluster created with {len(client.scheduler_info()['workers'])} workers")
print(f"   Dashboard: {client.dashboard_link}")
print(f"   Temp directory: {temp_dir}")


def benchmark_chunking_performance(
    ds: xr.Dataset,
    chunk_options: Dict[str, Dict[str, int]],
    operation: str = "mean"
) -> Dict[str, float]:
    """Benchmark performance of different chunking strategies."""
    
    print(f"\n⚡ PERFORMANCE BENCHMARK - {operation.upper()} OPERATION")
    print("=" * 50)
    
    results = {}
    
    for option_name, chunks in chunk_options.items():
        if chunks is None or 'error' in str(chunks):
            continue
            
        print(f"\n🏃 Testing: {option_name}")
        
        try:
            # Apply chunking
            chunked_ds = ds.chunk(chunks)
            
            # Select temperature variable for testing
            data_var = chunked_ds['temperature']
            
            # Define the operation
            if operation == "mean":
                computation = lambda: data_var.mean().compute()
            elif operation == "sum":
                computation = lambda: data_var.sum().compute()
            elif operation == "std":
                computation = lambda: data_var.std().compute()
            elif operation == "spatial_mean":
                computation = lambda: data_var.mean(['lat', 'lon']).compute()
            else:
                print(f"   Unknown operation: {operation}")
                continue
            
            # Warm up (first run can be slower)
            print("   Warming up...")
            try:
                _ = computation()
            except Exception as e:
                print(f"   Warmup failed: {e}")
                continue
            
            # Benchmark the operation
            print("   Running benchmark...")
            start_time = time.time()
            result = computation()
            elapsed_time = time.time() - start_time
            
            results[option_name] = elapsed_time
            print(f"   Result: {float(result):.6f}")
            print(f"   Time: {elapsed_time:.3f}s")
            
        except Exception as e:
            print(f"   ❌ Benchmark failed: {e}")
            results[option_name] = float('inf')
    
    return results


# Get valid chunking options (exclude errors)
valid_chunk_options = {
    k: v['chunks'] for k, v in analysis_results.items() 
    if 'error' not in v and v['chunks'] is not None
}

# Benchmark different operations
operations = ['mean', 'spatial_mean', 'std']
benchmark_results = {}

for operation in operations:
    print(f"\n🎯 Benchmarking {operation} operation...")
    benchmark_results[operation] = benchmark_chunking_performance(
        ds, valid_chunk_options, operation
    )

# Display benchmark summary
print("\n" + "=" * 60)
print("🏁 BENCHMARK SUMMARY")
print("=" * 60)

for operation, results in benchmark_results.items():
    print(f"\n{operation.upper()} Operation:")
    valid_results = {k: v for k, v in results.items() if v != float('inf')}
    if valid_results:
        best_time = min(valid_results.values())
        sorted_results = sorted(valid_results.items(), key=lambda x: x[1])
        
        for strategy, time_s in sorted_results:
            relative_speed = time_s / best_time
            print(f"   {strategy:20s} | {time_s:6.3f}s | {relative_speed:4.2f}x")
    else:
        print("   No valid results")

## Chunking Visualization

Let's visualize the chunk structure and performance results.

In [ ]:
if PLOTTING_AVAILABLE:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Create visualization of chunk sizes and performance
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Chunk sizes comparison
    strategies = []
    chunk_sizes = []
    colors = []
    
    for strategy, result in analysis_results.items():
        if 'error' not in result:
            strategies.append(strategy)
            chunk_sizes.append(result['avg_chunk_mb'])
            
            # Color code based on quality
            if '✅' in result['quality']:
                colors.append('green')
            elif '🟡' in result['quality']:
                colors.append('orange')
            else:
                colors.append('red')
    
    # Create bar plot
    bars = ax1.bar(range(len(strategies)), chunk_sizes, color=colors, alpha=0.7)
    ax1.set_xlabel('Chunking Strategy')
    ax1.set_ylabel('Average Chunk Size (MB)')
    ax1.set_title('Chunk Size Comparison')
    ax1.set_xticks(range(len(strategies)))
    ax1.set_xticklabels(strategies, rotation=45, ha='right')
    
    # Add optimal range shading
    ax1.axhspan(256, 512, alpha=0.2, color='green', label='Optimal Range (256-512 MB)')
    ax1.legend()
    
    # Add value labels on bars
    for bar, size in zip(bars, chunk_sizes):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{size:.1f}', ha='center', va='bottom', fontsize=8)
    
    # Plot 2: Performance comparison (using mean operation)
    if 'mean' in benchmark_results:
        perf_data = benchmark_results['mean']
        valid_perf = {k: v for k, v in perf_data.items() if v != float('inf')}
        
        if valid_perf:
            perf_strategies = list(valid_perf.keys())
            perf_times = list(valid_perf.values())
            
            # Normalize to show relative performance
            best_time = min(perf_times)
            relative_times = [t / best_time for t in perf_times]
            
            bars2 = ax2.bar(range(len(perf_strategies)), relative_times, 
                           color='skyblue', alpha=0.7)
            ax2.set_xlabel('Chunking Strategy')
            ax2.set_ylabel('Relative Performance (lower is better)')
            ax2.set_title('Performance Comparison (Mean Operation)')
            ax2.set_xticks(range(len(perf_strategies)))
            ax2.set_xticklabels(perf_strategies, rotation=45, ha='right')
            
            # Add value labels
            for bar, time_val in zip(bars2, relative_times):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                        f'{time_val:.2f}x', ha='center', va='bottom', fontsize=8)
    else:
        ax2.text(0.5, 0.5, 'No performance data available', 
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('Performance Comparison')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("📊 Plotting not available (matplotlib not installed)")
    print("   Install matplotlib to see chunk size and performance visualizations")

## Rechunking Demonstration

Use `rechunk_dataset()` from `dask_setup` to safely rechunk large datasets.
It uses the `rechunker` library when available and compatible, and automatically
falls back to `xarray.to_zarr()` on environments where rechunker is incompatible
with the installed xarray version (e.g. NCI Gadi's `analysis3` environments).

In [ ]:
from dask_setup import rechunk_dataset
import tempfile

def demonstrate_rechunking(ds, source_chunks, target_chunks, client, dask_tmp):
    """Demonstrate rechunking with rechunk_dataset()."""
    print("\n🔄 RECHUNKING DEMONSTRATION")
    print("=" * 40)

    source_ds = ds.chunk(source_chunks)
    print(f"Source chunks: {source_chunks}")
    print(f"Target chunks: {target_chunks}")

    # rechunk_dataset handles rechunker/xarray compatibility automatically.
    # If rechunker is incompatible (e.g. newer xarray requires zarr_format),
    # it falls back to native xarray.to_zarr() with no API change.
    ds_rechunked = rechunk_dataset(
        source_ds,
        target_chunks=target_chunks,
        client=client,
        dask_tmp=dask_tmp,
    )

    print(f"✅ Rechunked successfully")
    print(f"   New chunks: {dict(ds_rechunked.chunks)}")
    return ds_rechunked


# Usage (run inside a setup_dask_client() context):
# client, cluster, dask_tmp = setup_dask_client('io')
# ds_rechunked = demonstrate_rechunking(
#     ds,
#     source_chunks={'time': 1, 'lat': 90, 'lon': 180},
#     target_chunks={'time': 30, 'lat': 90, 'lon': 180},
#     client=client,
#     dask_tmp=dask_tmp,
# )
print("✅ rechunk_dataset() helper defined")
print("   • Uses rechunker when available and compatible")
print("   • Falls back to xarray.to_zarr() automatically")
print("   • Intermediate and output stores go to dask_tmp (fast local NVMe)")


## Chunking Best Practices Guide

Based on our analysis, here are the key best practices for xarray chunking:

In [ ]:
def print_chunking_best_practices():
    """Print comprehensive chunking best practices."""
    print("\n" + "=" * 60)
    print("💡 XARRAY CHUNKING BEST PRACTICES")
    print("=" * 60)
    
    print("""
🎯 OPTIMAL CHUNK SIZE RULE:
   • Target: 256-512 MB per chunk
   • Avoid: <100 MB (too much overhead) or >1 GB (memory issues)
   • Monitor: Actual memory usage vs estimates

📐 CHUNK SHAPE STRATEGIES:

   CPU-Intensive Workloads:
   • Square-ish chunks for better cache locality
   • Balance dimensions equally when possible
   • Example: (500, 500, 10) rather than (100, 100, 500)

   I/O-Intensive Workloads:
   • Preserve natural access patterns
   • Keep record dimensions (time) lightly chunked
   • Example: (365, 180, 90) for daily global data

   Balanced Workloads:
   • Mixed approach based on operation types
   • Consider downstream processing patterns

🔄 RECHUNKING STRATEGIES:

   Use .chunk() when:
   • Initial chunking for analysis
   • Simple chunk size adjustments
   • Interactive exploration

   Use rechunk_dataset() when:
   • Major chunk structure changes (writes to Zarr on fast local storage)
   • Large datasets (>1 GB) — handles rechunker/xarray compatibility
   • Need to persist rechunked data to disk for repeated access

⚠️  COMMON PITFALLS:

   BAD Chunking Examples:
   • Too small: (10, 10, 10) → ~8KB chunks (high overhead)
   • Too large: (5000, 5000, 100) → ~20GB chunks (memory issues)
   • Unbalanced: (1, 3600, 1800) → inefficient parallelization

   GOOD Chunking Examples:
   • Balanced: (240, 512, 512) → ~512MB chunks
   • Time-aware: (365, 180, 90) → preserves daily records
   • Analysis-optimized: (100, 500, 500) → good for spatial ops

🔧 CONFIGURATION EXAMPLES:

   # For large climate datasets
   ds.chunk({"time": 365, "lat": 180, "lon": 360})  # ~390MB chunks
   
   # For high-resolution spatial data
   ds.chunk({"time": 30, "y": 1000, "x": 1000})   # ~240MB chunks
   
   # For time series analysis
   ds.chunk({"time": -1, "station": 100})          # Keep time continuous

📊 MEMORY CALCULATION:
   chunk_bytes = chunk_elements × dtype_size
   
   Example: (500, 500, 10) float64 array
   = 500 × 500 × 10 × 8 bytes = 20MB (too small)
   
   Better: (500, 500, 100) float64 array  
   = 500 × 500 × 100 × 8 bytes = 200MB ✓ (acceptable)
""")

# Display the best practices
print_chunking_best_practices()

## Summary Comparison Table

Let's create a comprehensive comparison table of all our chunking strategies.

In [ ]:
def create_chunking_comparison_table(analysis_results, benchmark_results=None):
    """Create a comprehensive comparison table."""
    print("\n" + "=" * 80)
    print("📋 COMPREHENSIVE CHUNKING STRATEGY COMPARISON")
    print("=" * 80)
    
    # Prepare headers
    headers = ["Strategy", "Chunks", "Avg Size (MB)", "Total Chunks", "Quality"]
    
    # Add performance columns if available
    if benchmark_results and 'mean' in benchmark_results:
        headers.extend(["Mean Time (s)", "Relative Speed"])
    
    # Prepare data
    table_data = []
    
    # Get best performance for relative comparison
    best_time = None
    if benchmark_results and 'mean' in benchmark_results:
        valid_times = [t for t in benchmark_results['mean'].values() if t != float('inf')]
        if valid_times:
            best_time = min(valid_times)
    
    # Sort strategies by chunk size (optimal range first)
    sorted_strategies = sorted(
        analysis_results.items(),
        key=lambda x: abs(x[1].get('avg_chunk_mb', 1000) - 384)  # 384 = middle of optimal range
    )
    
    for strategy, data in sorted_strategies:
        if 'error' in data:
            continue
            
        # Format chunks dictionary
        chunks_str = ", ".join([f"{k}:{v}" for k, v in data['chunks'].items()])
        if len(chunks_str) > 25:
            chunks_str = chunks_str[:22] + "..."
        
        row = [
            strategy,
            chunks_str,
            f"{data['avg_chunk_mb']:.1f}",
            str(data['total_chunks']),
            data['quality']
        ]
        
        # Add performance data if available
        if benchmark_results and 'mean' in benchmark_results:
            if strategy in benchmark_results['mean']:
                time_val = benchmark_results['mean'][strategy]
                if time_val != float('inf') and best_time:
                    relative_speed = time_val / best_time
                    row.extend([f"{time_val:.3f}", f"{relative_speed:.2f}x"])
                else:
                    row.extend(["Failed", "N/A"])
            else:
                row.extend(["-", "-"])
        
        table_data.append(row)
    
    # Print table with proper formatting
    if table_data:
        # Calculate column widths
        col_widths = []
        for i in range(len(headers)):
            col_width = max(
                len(headers[i]),
                max(len(str(row[i])) for row in table_data if len(row) > i)
            )
            col_widths.append(col_width)
        
        # Print header
        header_row = " | ".join(headers[i].ljust(col_widths[i]) for i in range(len(headers)))
        print(f"| {header_row} |")
        print("|" + "|".join("-" * (w + 2) for w in col_widths) + "|")
        
        # Print data rows
        for row in table_data:
            # Pad row if needed
            while len(row) < len(headers):
                row.append("-")
            
            data_row = " | ".join(str(row[i]).ljust(col_widths[i]) for i in range(len(headers)))
            print(f"| {data_row} |")
    else:
        print("No valid results to display")
    
    # Add interpretation guide
    print("\n💡 Interpretation Guide:")
    print("   • Optimal chunk size: 256-512 MB")
    print("   • Quality indicators: ✅ Optimal, 🟡 Acceptable, 🔴 Problematic")
    print("   • Lower relative speed is better (1.0x = fastest)")
    print("   • Choose strategy based on your specific workload")

# Create the comprehensive comparison table
create_chunking_comparison_table(analysis_results, benchmark_results)

# Clean up the cluster
client.close()
cluster.close()
print("\n🧹 Cluster cleaned up")

## Personalized Recommendations

Based on our analysis, here are specific recommendations for your dataset:

In [ ]:
def generate_recommendations(analysis_results, benchmark_results=None):
    """Generate personalized chunking recommendations."""
    print("\n" + "=" * 60)
    print("🎯 PERSONALIZED RECOMMENDATIONS")
    print("=" * 60)
    
    # Find the best strategies
    optimal_strategies = []
    acceptable_strategies = []
    
    for strategy, data in analysis_results.items():
        if 'error' not in data:
            if '✅' in data['quality']:
                optimal_strategies.append((strategy, data))
            elif '🟡' in data['quality']:
                acceptable_strategies.append((strategy, data))
    
    # Find fastest strategy if performance data is available
    fastest_strategy = None
    if benchmark_results and 'mean' in benchmark_results:
        valid_times = {k: v for k, v in benchmark_results['mean'].items() if v != float('inf')}
        if valid_times:
            fastest_strategy = min(valid_times.items(), key=lambda x: x[1])
    
    print(f"\n📊 For your dataset (shape: {ds.temperature.shape}, size: {ds.nbytes/(1024**2):.1f} MB):")
    
    if optimal_strategies:
        print(f"\n✅ OPTIMAL STRATEGIES ({len(optimal_strategies)} found):")
        for strategy, data in optimal_strategies:
            print(f"   • {strategy}: {data['chunks']} → {data['avg_chunk_mb']:.1f} MB chunks")
            
            # Add context about when to use this strategy
            if 'cpu' in strategy:
                print(f"     → Use for: Heavy computations, mathematical operations")
            elif 'io' in strategy:
                print(f"     → Use for: File I/O operations, data loading/saving")
            elif 'balanced' in strategy:
                print(f"     → Use for: Mixed workloads, general analysis")
            elif 'time_continuous' in strategy:
                print(f"     → Use for: Time series analysis, temporal operations")
            elif 'spatial_focus' in strategy:
                print(f"     → Use for: Spatial analysis, geographical operations")
    else:
        print("\n🟡 No optimal strategies found. Acceptable alternatives:")
    
    if acceptable_strategies:
        if optimal_strategies:
            print(f"\n🟡 ACCEPTABLE ALTERNATIVES ({len(acceptable_strategies)} found):")
        for strategy, data in acceptable_strategies[:3]:  # Show top 3
            print(f"   • {strategy}: {data['chunks']} → {data['avg_chunk_mb']:.1f} MB chunks")
    
    # Performance recommendation
    if fastest_strategy:
        strategy_name, best_time = fastest_strategy
        print(f"\n⚡ FASTEST STRATEGY: {strategy_name} ({best_time:.3f}s for mean operation)")
        if strategy_name in analysis_results:
            print(f"   Configuration: {analysis_results[strategy_name]['chunks']}")
    
    # Specific use case recommendations
    print(f"\n🔧 SPECIFIC USE CASE RECOMMENDATIONS:")
    
    print(f"\n   For Time Series Analysis:")
    if 'time_continuous' in analysis_results:
        time_config = analysis_results['time_continuous']['chunks']
        print(f"     ds.chunk({time_config})")
    else:
        print(f"     ds.chunk({{'time': -1, 'lat': 90, 'lon': 180}})  # Keep time continuous")
    
    print(f"\n   For Spatial Analysis:")
    if 'spatial_focus' in analysis_results:
        spatial_config = analysis_results['spatial_focus']['chunks']
        print(f"     ds.chunk({spatial_config})")
    else:
        print(f"     ds.chunk({{'time': 30, 'lat': -1, 'lon': -1}})  # Keep spatial dims continuous")
    
    print(f"\n   For General Computing:")
    if optimal_strategies:
        best_general = optimal_strategies[0][1]['chunks']
        print(f"     ds.chunk({best_general})")
    elif acceptable_strategies:
        best_general = acceptable_strategies[0][1]['chunks']
        print(f"     ds.chunk({best_general})")
    
    # Memory considerations
    print(f"\n💾 MEMORY CONSIDERATIONS:")
    print(f"   • Your dataset: {ds.nbytes/(1024**3):.2f} GB total")
    print(f"   • Recommended worker memory: 4-8 GB per worker")
    print(f"   • Consider using reserve_mem_gb=20-40 for safety")

# Generate recommendations
generate_recommendations(analysis_results, benchmark_results)

## Conclusion

You've now learned the comprehensive approach to xarray chunking optimization! Here's what we covered:

### Key Takeaways:
1. **Optimal Chunk Size**: Target 256-512 MB per chunk for best performance
2. **Workload-Specific Strategies**: CPU, I/O, and balanced approaches
3. **Performance Impact**: Proper chunking can significantly improve computation speed
4. **Memory Management**: Balance chunk size with available memory
5. **Rechunking**: Use `rechunk_dataset()` for major structural changes — it handles rechunker/xarray compatibility automatically

### Best Practices Recap:
- **Measure first**: Always analyze your data and benchmark different strategies
- **Consider your workload**: Choose chunking strategy based on operations you'll perform
- **Monitor memory**: Watch for spills and memory pressure in the dashboard
- **Start conservative**: Begin with smaller chunks and scale up
- **Test with real data**: Synthetic examples may not capture your data's characteristics

### Next Steps:
- Apply these techniques to your own datasets
- Experiment with different chunk configurations
- Monitor performance in the Dask dashboard
- Use `rechunk_dataset()` for large-scale rechunking — it falls back gracefully when rechunker is incompatible with your xarray version
- Integrate optimal chunking into your data processing pipelines

Happy chunking with dask_setup! 🧩✨